In [ ]:
import sys
from pathlib import Path
import json
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timezone
import re
from tqdm import tqdm  
import torch
import gc

# Setup project paths
project_root = Path.cwd()
sys.path.append(str(project_root))
sys.path.append(str(project_root / "src"))

from src.config import load_config
from models.load_model import ModelLoader
from generators.program_generator import GenerationPipeline

In [ ]:
sns.set_theme(style="whitegrid")

In [ ]:
dev_json_path = "data/raw/Spider/spider_data/dev.json"
db_root_path = "data/raw/Spider/spider_data/database/"

print(f"Loading dataset from: {dev_json_path}")

try:
    with open(dev_json_path, "r", encoding="utf-8") as f:
        dev_data = json.load(f)
        df_spider = pd.DataFrame(dev_data)
    print(f"Successfully loaded {len(df_spider)} samples from Spider dataset!")
except Exception as e:
    print(f"Error loading dataset: {e}")
    sys.exit(1)

In [ ]:
def execute_sql_query(db_path, query):
    """Executes an SQL query on a SQLite database and returns the result set."""
    if not os.path.exists(db_path):
        return {"success": False, "error": "Database file not found", "result": None}

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(query)
        result = cursor.fetchall()
        conn.close()
        return {"success": True, "error": None, "result": result}
    except Exception as e:
        return {"success": False, "error": str(e), "result": None}

In [ ]:
# ==========================================
# PART 1: SQL Execution Evaluation
# ==========================================
sample_limit = min(200000, len(df_spider))
evaluated_records = []
predicted_records = []

print(f"\nRunning execution evaluation on {sample_limit} samples...")

# Optional: You can also wrap this loop in tqdm if you want a progress bar here too!
for idx, row in df_spider.head(sample_limit).iterrows():
    db_id = row.get("db_id")
    gold_query = row.get("query")
    question = row.get("question")

    db_file_path = os.path.join(db_root_path, db_id, f"{db_id}.sqlite")
    
    # NOTE: Replace 'gold_query' with your actual model prediction when ready
    predicted_query = gold_query 

    gold_exec = execute_sql_query(db_file_path, gold_query)
    pred_exec = execute_sql_query(db_file_path, predicted_query)

    is_execution_correct = False
    if gold_exec["success"] and pred_exec["success"]:
        try:
            is_execution_correct = set(gold_exec["result"]) == set(pred_exec["result"])
        except Exception:
            is_execution_correct = gold_exec["result"] == pred_exec["result"]

    is_exact_match = (predicted_query.strip().lower() == gold_query.strip().lower())

    evaluated_records.append({
        "query_id": f"sample_{idx}",
        "database_id": db_id,
        "question": question,
        "gold_query": [gold_query]
    })

    predicted_records.append({
        "predicted_query": predicted_query,
        "exact_match": is_exact_match,
        "execution_accuracy": is_execution_correct,
        "execution_error": pred_exec["error"]
    })

df_results = pd.DataFrame(evaluated_records)
predicted_df = pd.DataFrame(predicted_records)

total_samples = len(df_results)
response_acc = (predicted_df["exact_match"].sum() / total_samples) * 100
execution_acc = (predicted_df["execution_accuracy"].sum() / total_samples) * 100

print("\n--- Real Execution Evaluation Summary ---")
print(f"Total Evaluated: {total_samples}")
print(f"Response Accuracy (Exact Match): {response_acc:.2f}%")
print(f"Execution Accuracy: {execution_acc:.2f}%")

# Save summary JSON
mongo_document = {
    "detailed_results": df_results.to_dict(orient="records")
}

output_filename = "spider_real_execution_mongo.json"
with open(output_filename, "w", encoding="utf-8") as json_file:
    json.dump(mongo_document, json_file, indent=4)

print(f"\nSuccessfully generated MongoDB-ready JSON document: '{output_filename}'")

In [ ]:
# ==========================================
# PART 2: Load Model & Initialize Pipeline
# ==========================================
print("\n🔄 Loading model and tokenizer (this may take a minute)...")
config = load_config("configs/config.yaml")
model_loader = ModelLoader(config)
models, tokenizer = model_loader.load_models()

base_gen_pipeline = GenerationPipeline(
    model=models["base"],
    tokenizer=tokenizer,
    config=config.generation
)
print("✅ base_gen_pipeline is ready!")

In [ ]:
# ==========================================
# PART 3: SQL-to-MongoDB Converter (Memory & Speed Optimized)
# ==========================================
def convert_sql_to_mongo_via_llm(sql_query: str, pipeline) -> str:
    """Uses an LLM generation pipeline to translate an SQL query into MongoDB syntax."""

    # 1. Ultra-short 1-shot prompt to minimize KV cache and generation time
    prompt = f"""SQL: SELECT count(*) FROM singer
MongoDB: db.singer.count()

SQL: {sql_query}
MongoDB: db."""

    try:
        # 2. Temporarily force a hard limit on generation length for this specific task
        original_max_length = getattr(pipeline.config, "max_length", 256)
        pipeline.config.max_length = 64  # MongoDB queries are very short

        raw_output = pipeline.generate(prompt)

        # Restore original config
        pipeline.config.max_length = original_max_length

        # 3. Clean up markdown formatting
        cleaned = raw_output.replace("```javascript", "").replace("```json", "").replace("```js", "").replace("```", "").strip()

        # Ensure it starts with db.
        if not cleaned.startswith("db."):
            lines = cleaned.split('\n')
            for line in lines:
                if line.strip().startswith("db."):
                    cleaned = line.strip()
                    break
            else:
                cleaned = "db." + cleaned # Fallback prepend

        return cleaned

    except Exception as e:
        return f"// Error: {str(e)}"

    finally:
        # 4. AGGRESSIVE MEMORY CLEARING: Prevents GPU/MPS spikes across loop iterations
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()
        gc.collect()

In [ ]:
# ==========================================
# PART 4: Run Conversion & Save Results
# ==========================================
conversion_sample_limit = min(10, len(df_spider))
mongodb_conversion_results = []
successful_conversions = 0

print(f"\n🔄 Running SQL-to-MongoDB LLM conversion on {conversion_sample_limit} samples...")

# Wrap the iterator with tqdm to show a progress bar
for idx, row in tqdm(df_spider.head(conversion_sample_limit).iterrows(),
                     total=conversion_sample_limit,
                     desc="Converting SQL to MongoDB"):
    db_id = row.get("db_id")
    raw_sql_query = row.get("query")
    question = row.get("question")

    generated_mongo = convert_sql_to_mongo_via_llm(raw_sql_query, base_gen_pipeline)

    status = "success" if generated_mongo.strip().startswith("db.") else "failed/partial"
    if status == "success":
        successful_conversions += 1

    mongodb_conversion_results.append({
        "query_id": f"sample_{idx}",
        "database_id": db_id,
        "question": question,
        "generated_mongodb_query": [generated_mongo]
    })

success_rate = round((successful_conversions / conversion_sample_limit) * 100, 2) if conversion_sample_limit > 0 else 0.0

mongo_conversion_doc = {
    "detailed_results": mongodb_conversion_results
}

conversion_output_filename = "qwen_spider_mongodb_conversion.json"
with open(conversion_output_filename, "w", encoding="utf-8") as json_file:
    json.dump(mongo_conversion_doc, json_file, indent=4, ensure_ascii=False)

print(f"\n✅ Successfully generated MongoDB-ready JSON document: '{conversion_output_filename}'")
print(f"📊 Summary: {successful_conversions}/{conversion_sample_limit} successful conversions ({success_rate}%)")